# Physics Simulation and Interaction
### Go to "Direct import trained model" from file to skip training

In [1]:
import copy, math, os, sys, logging, threading
from typing import List, Tuple
from pathlib import Path


import numpy as np
import torch
import kaolin as kal

from IPython.display import display
from ipywidgets import Button, HBox, VBox

logging.basicConfig(level=logging.INFO, stream=sys.stdout)
logging.getLogger('kaolin.physics').setLevel(logging.DEBUG)

sys.path.append(str(Path("...")))
from common_path import COMMON_DATA_DIR

def sample_mesh_path(fname):
    return os.path.join(COMMON_DATA_DIR, 'meshes', fname)

def print_tensor(t, name='', **kwargs):
    print(kal.utils.testing.tensor_info(t, name=name, **kwargs))
    

Warp 1.7.2 initialized:
   CUDA Toolkit 12.8, Driver 12.8
   Devices:
     "cpu"      : "AMD64 Family 26 Model 68 Stepping 0, AuthenticAMD"
     "cuda:0"   : "NVIDIA GeForce RTX 4090" (24 GiB, sm_89, mempool enabled)
   Kernel cache:
     C:\Users\sztkk\AppData\Local\NVIDIA\warp\Cache\1.7.2


## Loading Geometry

In [3]:
# Import and triangulate to enable rasterization; move to GPU
# mesh = kal.io.import_mesh(sample_mesh_path('armchair.obj'), triangulate=True).cuda()
mesh = kal.io.import_mesh(sample_mesh_path('prototype_i_glb_format_-_gislingebad.glb'), triangulate=True).cuda()
# Normalize so it is easy to set up default camera
mesh.vertices = kal.ops.pointcloud.center_points(mesh.vertices.unsqueeze(0), normalize=True).squeeze(0) 
orig_vertices = mesh.vertices.clone()  # Also save original undeformed vertices
print(mesh)

SurfaceMesh object with batching strategy NONE
            vertices: [23031, 3] (torch.float32)[cuda:0]  
               faces: [29035, 3] (torch.int64)[cuda:0]  
                 uvs: [23031, 2] (torch.float32)[cuda:0]  
        face_uvs_idx: [29035, 3] (torch.int64)[cuda:0]  
      vertex_normals: [23031, 3] (torch.float32)[cuda:0]  
     vertex_tangents: [23031, 3] (torch.float32)[cuda:0]  
material_assignments: [29035] (torch.int16)[cuda:0]  
           materials: list of length 2
       face_vertices: if possible, computed on access from: (faces, vertices)
        face_normals: if possible, computed on access from: (normals, face_normals_idx) or (vertex_normals, faces) or (vertices, faces)
            face_uvs: if possible, computed on access from: (uvs, face_uvs_idx)
       vertex_colors: if possible, computed on access from: (faces, face_colors)
     vertex_features: if possible, computed on access from: (faces, face_features)
       face_tangents: if possible, computed on acces

e:\MINICONDA\envs\kaolin\lib\site-packages\kaolin\io\gltf.py:285: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ..\torch\csrc\utils\tensor_new.cpp:1468.)
  output = torch.frombuffer(


## Visualizing Geometry

Because we are working with a mesh, Kaolin already comes with an easy render function. To use any rendering function (such as a Gaussan splat renderer),
simply define the following rendering functions that take `Camera` as input.

In [2]:
def render(in_cam):
    active_pass=kal.render.easy_render.RenderPass.render
    render_res = kal.render.easy_render.render_mesh(in_cam, mesh)

    # use white background
    img = render_res[active_pass]
    background_mask = (render_res[kal.render.easy_render.RenderPass.face_idx] < 0).bool()
    img2 = torch.clamp(img, 0, 1)[0]
    img2[background_mask[0]] = 1
    final = (img2 * 255.).to(torch.uint8)
    return {"img":final, "face_idx": render_res[kal.render.easy_render.RenderPass.face_idx].squeeze(0).unsqueeze(-1)}

# faster low-res render during mouse motion
def fast_render(in_cam, factor=8):
    lowres_cam = copy.deepcopy(in_cam)
    lowres_cam.width = in_cam.width // factor
    lowres_cam.height = in_cam.height // factor
    return render(lowres_cam)

resolution = 256
camera = kal.render.easy_render.default_camera(resolution).cuda()
rest_state_viz = kal.visualize.IpyTurntableVisualizer(
    resolution, resolution, copy.deepcopy(camera), render, fast_render=fast_render,
    max_fps=24, world_up_axis=1)
rest_state_viz.show()

e:\MINICONDA\envs\kaolin\lib\site-packages\torch\functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ..\aten\src\ATen\native\TensorShape.cpp:3484.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Canvas(height=256, width=256)

Output()

## Sample Geometry

To enable simulation we need point samples (vertices in this case), and physical material parameters per point. Let's set this up.

In [2]:
# Physics material parameters
soft_youngs_modulus = 1e5
hard_youngs_modulus = 1e7
poisson_ratio = 0.45
rho = 100  # kg/m^3
approx_volume = 1  # m^3

# Point samples
pts = mesh.vertices
yms = torch.full((pts.shape[0],), soft_youngs_modulus, device="cuda")
prs = torch.full((pts.shape[0],), poisson_ratio, device="cuda")
rhos = torch.full((pts.shape[0],), rho, device="cuda")

In [4]:
import k3d

particle_plot = k3d.plot(camera_auto_fit=False)
particle_plot += k3d.points(pts.cpu().detach().numpy(), point_size=0.005)
# Rotate the camera so that y is the vertical axis
particle_plot.camera = [1, 0, 1, 0, 0, 0, 0, 1, 0]  # (eye_x, eye_y, eye_z, target_x, target_y, target_z, up_x, up_y, up_z)
particle_plot.grid_visible = False
particle_plot.axes_helper = False
particle_plot.display()

Output()

## Create and Train a SimplicitsObject

In [ ]:
from physics.easy_api import SimplicitsObjectTest

# Initialize and train a object
sim_obj = SimplicitsObjectTest(pts, yms, prs, rhos, torch.tensor([approx_volume], dtype=torch.float32, device="cuda"), num_handles=5)
print('Training simplicits object. This will take 2-3min. ')
sim_obj.train(num_steps=20000, le_coeff=0.1)#0) # do nothing if num_handles = 0
print('Object ready to simulate.')

Training simplicits object. This will take 2-3min. 
INFO:easy_api_test:Training step: 0, le: 1013.33349609375, lo: 50645256.0
INFO:easy_api_test:Training step: 1000, le: 370.72637939453125, lo: 80458.125
INFO:easy_api_test:Training step: 2000, le: 1351.664794921875, lo: 43295.83984375
INFO:easy_api_test:Training step: 3000, le: 2040.8695068359375, lo: 2041.6639404296875
INFO:easy_api_test:Training step: 4000, le: 2595.111328125, lo: 5072.1259765625
INFO:easy_api_test:Training step: 5000, le: 2073.33349609375, lo: 5266.0263671875
INFO:easy_api_test:Training step: 6000, le: 2803.054443359375, lo: 3238.9345703125
INFO:easy_api_test:Training step: 7000, le: 1637.802001953125, lo: 2117.47021484375
INFO:easy_api_test:Training step: 8000, le: 2371.991455078125, lo: 3164.01171875
INFO:easy_api_test:Training step: 9000, le: 2720.79296875, lo: 5607.53125
INFO:easy_api_test:Training step: 10000, le: 2158.073974609375, lo: 3167.866943359375
INFO:easy_api_test:Training step: 11000, le: 1794.5559082

In [7]:
# Optionally, you can save/load this pre-trained object
# sim_obj.save_model('./tmp/chair_simplicit.pt')
sim_obj.save_model('./tmp/ship.pt')

### visualize training weight

In [11]:
import matplotlib.colors as mcolors

def scalar_to_rgb(s_normalized, mincolorstr, maxcolorstr, minval=None, maxval=None):
    cmap = mcolors.LinearSegmentedColormap.from_list("", [mincolorstr, maxcolorstr])

    if(np.sum(s_normalized)==0):
        return cmap(s_normalized)[:, 0:3]*255
    else:
        if minval == None:
            minval = np.min(s_normalized)
        if maxval == None:
            maxval = np.max(s_normalized)
        s_normalized = (s_normalized - np.min(s_normalized)) / (maxval - minval)
        colors_rgb = cmap(s_normalized)[:, 0:3]*255
        return colors_rgb

def rgb2hex(rgb):
    str_res = "0x{0:02x}{1:02x}{2:02x}".format(int(rgb[0]), int(rgb[1]), int(rgb[2]))
    return int(str_res, 16)

# _lbs_weight = so_model(so_pts)
_lbs_weight = sim_obj.model(sim_obj.pts)

def visualize_mode(viz_mode, _lbs_weight):
    rgb = scalar_to_rgb(_lbs_weight[:,viz_mode].detach().cpu().numpy(), "blue", "red")
    colors = [rgb2hex(rgb[xx, :]) for xx in range(rgb.shape[0])]
    return colors

# Viewing 3 of the weight functions
# Plot more to view more
weight_plot = k3d.plot(camera_auto_fit=False)
weight_plot.camera = [1, 0, 1, 0, 0, 0, 0, 1, 0]  # (eye_x, eye_y, eye_z, target_x, target_y, target_z, up_x, up_y, up_z)
weight_plot.grid_visible = False
weight_plot.axes_helper = False
# weight_plot += k3d.points(sim_obj.pts.detach().cpu().numpy(), colors=visualize_mode(0,_lbs_weight),  point_size=0.005)
# weight_plot += k3d.points(sim_obj.pts.detach().cpu().numpy(), colors=visualize_mode(4,_lbs_weight),  point_size=0.005)
weight_plot += k3d.points(sim_obj.pts.detach().cpu().numpy(), colors=visualize_mode(2,_lbs_weight),  point_size=0.005)
weight_plot.display()

Output()

# Direct import trained model from file

In [ ]:
from physics.easy_api import SymplecticObject

import copy, math, os, sys, logging, threading
from typing import List, Tuple
from pathlib import Path


import numpy as np
import torch
import kaolin as kal

from IPython.display import display
from ipywidgets import Button, HBox, VBox

logging.basicConfig(level=logging.INFO, stream=sys.stdout)
logging.getLogger('kaolin.physics').setLevel(logging.DEBUG)

sys.path.append(str(Path("..")))
from common_path import COMMON_DATA_DIR

def sample_mesh_path(fname):
    return os.path.join(COMMON_DATA_DIR, 'meshes', fname)

def print_tensor(t, name='', **kwargs):
    print(kal.utils.testing.tensor_info(t, name=name, **kwargs))

# Import and triangulate to enable rasterization; move to GPU
# mesh = kal.io.import_mesh(sample_mesh_path('armchair.obj'), triangulate=True).cuda()
mesh = kal.io.import_mesh(sample_mesh_path('prototype_i_glb_format_-_gislingebad.glb'), triangulate=True).cuda()
# Normalize so it is easy to set up default camera
mesh.vertices = kal.ops.pointcloud.center_points(mesh.vertices.unsqueeze(0), normalize=True).squeeze(0) 
orig_vertices = mesh.vertices.clone()  # Also save original undeformed vertices
print(mesh)

# Physics material parameters
soft_youngs_modulus = 1e5
hard_youngs_modulus = 1e7
poisson_ratio = 0.45
rho = 100  # kg/m^3
approx_volume = 1  # m^3

# Point samples
pts = mesh.vertices
yms = torch.full((pts.shape[0],), soft_youngs_modulus, device="cuda")
prs = torch.full((pts.shape[0],), poisson_ratio, device="cuda")
rhos = torch.full((pts.shape[0],), rho, device="cuda")

# load object from file
sim_obj = SymplecticObject(pts, yms, prs, rhos, torch.tensor([approx_volume], dtype=torch.float32, device="cuda"), num_handles=5)
# sim_obj.load_model('./tmp/chair_simplicit.pt')  # Load the pre-trained object
sim_obj.load_model('./tmp/ship.pt')  # Load the pre-trained object

def render(in_cam):
    active_pass=kal.render.easy_render.RenderPass.render
    render_res = kal.render.easy_render.render_mesh(in_cam, mesh)

    # use white background
    img = render_res[active_pass]
    background_mask = (render_res[kal.render.easy_render.RenderPass.face_idx] < 0).bool()
    img2 = torch.clamp(img, 0, 1)[0]
    img2[background_mask[0]] = 1
    final = (img2 * 255.).to(torch.uint8)
    return {"img":final, "face_idx": render_res[kal.render.easy_render.RenderPass.face_idx].squeeze(0).unsqueeze(-1)}

# faster low-res render during mouse motion
def fast_render(in_cam, factor=8):
    lowres_cam = copy.deepcopy(in_cam)
    lowres_cam.width = in_cam.width // factor
    lowres_cam.height = in_cam.height // factor
    return render(lowres_cam)

resolution = 256
# camera = kal.render.easy_render.default_camera(resolution).cuda()
# rest_state_viz = kal.visualize.IpyTurntableVisualizer(
#     resolution, resolution, copy.deepcopy(camera), render, fast_render=fast_render,
#     max_fps=24, world_up_axis=1)
# rest_state_viz.show()

Warp 1.7.2 initialized:
   CUDA Toolkit 12.8, Driver 12.8
   Devices:
     "cpu"      : "AMD64 Family 26 Model 68 Stepping 0, AuthenticAMD"
     "cuda:0"   : "NVIDIA GeForce RTX 4090" (24 GiB, sm_89, mempool enabled)
   Kernel cache:
     C:\Users\sztkk\AppData\Local\NVIDIA\warp\Cache\1.7.2
SurfaceMesh object with batching strategy NONE
            vertices: [23031, 3] (torch.float32)[cuda:0]  
               faces: [29035, 3] (torch.int64)[cuda:0]  
                 uvs: [23031, 2] (torch.float32)[cuda:0]  
        face_uvs_idx: [29035, 3] (torch.int64)[cuda:0]  
      vertex_normals: [23031, 3] (torch.float32)[cuda:0]  
     vertex_tangents: [23031, 3] (torch.float32)[cuda:0]  
material_assignments: [29035] (torch.int16)[cuda:0]  
           materials: list of length 2
       face_vertices: if possible, computed on access from: (faces, vertices)
        face_normals: if possible, computed on access from: (normals, face_normals_idx) or (vertex_normals, faces) or (vertices, faces)
    

e:\MINICONDA\envs\kaolin\lib\site-packages\kaolin\io\gltf.py:285: UserWarning: The given buffer is not writable, and PyTorch does not support non-writable tensors. This means you can write to the underlying (supposedly non-writable) buffer using the tensor. You may want to copy the buffer to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ..\torch\csrc\utils\tensor_new.cpp:1468.)
  output = torch.frombuffer(


## Setting up Simulation

Once the object is trained, we can set up any number of simulated scenes with that object. 

Let's set up a default scene that includes floor and can cause falling under gravity.


In [ ]:
from physics.easy_api import SymplecticScene

# scene1 = kal.physics.simplicits.SimplicitsScene() # default scene
scene1 = SymplecticScene(timestep=0.003)
scene1.max_newton_steps = 3 # Smaller NM iteration count results in faster sims, but poorer convergence

# Add an object to the scene
obj_idx = scene1.add_object(sim_obj)

# Add gravity to the scene
scene1.set_scene_gravity(acc_gravity=torch.tensor([0, 9.8, 0]))
# Add floor to the scene
scene1.set_scene_floor(floor_height=-1, floor_axis=1, floor_penalty=10000)
# Make object even softer
scene1.sim_obj_dict[obj_idx].set_materials(yms=torch.tensor(1e5, device="cuda", dtype=torch.float32))

## Running and Visualizing Simulation

We will run the simulation, changing object point locations (mesh vertices in this case) at every timestep and will visualize at the same time.

Try changing camera position with your mouse and running the simulation to view it from another angle.

### Use k3d for better framerate

In [ ]:
# Adjust range() to simulate different time step

states = []
for s in range(3000):
    z = scene1.run_sim_step()
    # print(".", end="")
    # mesh.vertices = scene1.get_object_deformed_pts(obj_idx).squeeze()
    # standard_viz.render_update()

    states.append(z)

In [7]:
from scipy.spatial import Delaunay
import k3d
import ipywidgets as widgets

def generate_ground_plane(plane):
    # Define the size of the ground plane
    x_size = 1
    y_size = 1
    num_points_x = 30
    num_points_y = 30
    
    # Generate grid points
    x = np.linspace(-x_size/2, x_size/2, num_points_x)
    y = np.linspace(-y_size/2, y_size/2, num_points_y)
    X, Y = np.meshgrid(x, y)
    
    # Generate vertices
    vertices = np.column_stack([X.flatten(), plane * np.ones_like(X.flatten()), Y.flatten()])
    vertices += +1e-3*np.random.rand(vertices.shape[0], vertices.shape[1])
    # Perform Delaunay triangulation
    tri = Delaunay(vertices)
    return torch.tensor(vertices), torch.tensor(tri.simplices)

v, f = generate_ground_plane(-1)
def reconstruct_deformed_positions(sample_pts, weights, batch_transforms):
    # sample_pts: (N, 3)
    # weights: (N, H)
    # batch_transforms: (B, H, 3, 4)  # assuming B = 1 or broadcast
    
    N, H = weights.shape
    B = batch_transforms.shape[0]
    device = sample_pts.device

    # Add 1 to each point to make it homogeneous (N, 4)
    sample_pts_h = torch.cat([sample_pts, torch.ones_like(sample_pts[..., :1])], dim=-1)  # (N, 4)

    # Expand dims to align for broadcasting
    pts_exp = sample_pts_h.unsqueeze(1).expand(-1, H, -1)        # (N, H, 4)
    trans_exp = batch_transforms[0] if B == 1 else batch_transforms  # (H, 3, 4)
    
    # Apply transform: (N, H, 3) = sum over i: w_i * (T_i @ x)
    transformed = torch.matmul(trans_exp, pts_exp.transpose(-1, -2)).transpose(-1, -2)  # (N, H, 3)
    
    # Apply LBS weights
    x_deformed = torch.sum(weights.unsqueeze(-1) * transformed, dim=1)  # (N, 3)
    
    return x_deformed
# Function to create points
def create_points(t):
    z = states[int(t)]
    x = scene1.sim_obj_dict[obj_idx].B @ z + scene1.sim_obj_dict[obj_idx].x0_flat
    # print(torch.norm(x - x0_flat))
    scene_verts = x.reshape(-1,3).unsqueeze(0).cpu().detach()
    return scene_verts.cpu().detach().numpy()

# Create a plot
plot = k3d.plot(camera_auto_fit=False)
plot.grid_visible = False
plot.axes_helper = False
floor_plot = k3d.points(v.cpu().detach().numpy(), point_size=0.01, color=0x00ff00, shader='3d')
plot += floor_plot
# Initial set of points
initial_points = create_points(0)
points_plot = k3d.points(initial_points, point_size=0.02, color=0x0000ff, shader='3d')
plot += points_plot

# Rotate the camera so that y is the vertical axis
plot.camera = [3, 1, 0, 0, 0, 0, 0, 1, 0]  # (eye_x, eye_y, eye_z, target_x, target_y, target_z, up_x, up_y, up_z)


# Display the plot
plot.display()

# Define the function to update the points
def update_points(TIMESTEP):
    new_points = create_points(TIMESTEP)
    points_plot.positions = new_points.astype(np.float32)

# Create a slider
# slider = widgets.FloatSlider(min=0, max=len(states), step=1, value=0)
slider = widgets.FloatSlider(min=0, max=len(states), step=1, value=0)

# Link the slider to the update function
widgets.interactive(update_points, TIMESTEP=slider)

# Display the slider
display(slider)

Output()

FloatSlider(value=0.0, description='TIMESTEP', max=3000.0, step=1.0)

### Use default randerer 

In [ ]:
mesh.vertices = orig_vertices

new_camera = kal.render.easy_render.default_camera(resolution).cuda()
new_camera.extrinsics.move_forward(1)
standard_viz = kal.visualize.IpyTurntableVisualizer(
    resolution, resolution, copy.deepcopy(new_camera), render, fast_render=fast_render,
    max_fps=10, world_up_axis=1)
standard_viz.render_update()

def run_sim():
    global mesh
    
    scene1.reset()
    for s in range(2000):
        scene1.run_sim_step()
        print(".", end="")
        mesh.vertices = scene1.get_object_deformed_pts(obj_idx).squeeze()
        standard_viz.render_update()

def start_simulation(b):
    global button
    button.disabled = True
    with standard_viz.out:
        run_sim()
    button.disabled = False
        
button = Button(description='Run Sim')
button.on_click(start_simulation)
display(HBox([standard_viz.canvas, button]), standard_viz.out)

Output()

## Run Interactive Simulation! Use Shift + Click and pull on the object

### The default renderer is unstable, low framerate
##### restart kernel to reset GPU memory after each interactivate force input 

Now, let's enable pulling on object parts during simulation.
To interact, simply press "SHIFT" and click on the mesh and drag.

In [ ]:
def convert_offset_to_world_coords(kal_cam, point: torch.Tensor, offset: torch.Tensor):
    """ Given a point in 3D and a 2D offset in NDC coordinates, maps the offset to "world coordinate" units.
    In other words: recomputes the world coordinates had it been translated from point an "offset" amount in NDC
    space (-1 to 1).
    ps_camera: The current camera used to transform coords from world space -> camera view space -> NDC space
    point: The 3D point we translate from
    offset: A 2D offset in NDC coordinates
    """
    if point.ndim == 1:
        point = point[None]
    # Ask kaolin about the camera up and right axes in world coordinates, and move along them an "offset" amount.
    # offset is given in "post projected" NDC coordinates, so the amount isn't exactly camera-space units.
    # However, the important part is we don't move along the camera-forward axis.
    fov_x = kal_cam.fov_x
    fov_y = kal_cam.fov_y
    depth = torch.linalg.norm(point - kal_cam.cam_pos())
    offset[0] = offset[0]/1000#*torch.tan(fov_x)
    offset[1] = offset[1]/1000#*torch.tan(fov_y)
    
    translated_point = point.clone()
    translated_point += kal_cam.cam_right().squeeze(-1) * offset[0]
    translated_point += kal_cam.cam_up().squeeze(-1) * offset[1]
    return translated_point
    
def find_closest_3d_points(query_3d_pts, object_3d_pts, radius, k=10):
    """ Finds points from object_3d_pts to query_3d_pts
        Pts should be within radius and limited to a number k
    """
    # Define the radius
    r = radius
    # Calculate pairwise distances
    dists = torch.cdist(query_3d_pts, object_3d_pts)
    # Find points within radius r
    within_radius = dists <= r
    
    # Get the indices of object_pts within radius r of any query_pts
    indices_within_radius = torch.nonzero(within_radius, as_tuple=True)[1]
    if k >= indices_within_radius.shape[0]:
        return indices_within_radius, object_3d_pts[indices_within_radius]
    else:
        # Flatten distances tensor for sorting
        dists_flat = dists[within_radius]
        # Sort the distances and select top k
        sorted_indices = torch.argsort(dists_flat)[:k]
        # Select the top k indices
        top_k_indices = indices_within_radius[sorted_indices]
        # Get the points that are within the radius and in the top k closest
        points_within_radius_top_k = object_3d_pts[top_k_indices]
        # Get the points that are within the radius
        return top_k_indices, points_within_radius_top_k


active_face = -1
sim_history = []
mesh.vertices = orig_vertices
mouse_clicked = 0
scene1.sim_obj_dict[obj_idx].reset_sim_state()

def boundary_func(pts):
    # Extract the z-coordinates (height) of the points
    heights = pts[:, 1]
    # Determine the minimum and maximum z-coordinates
    z_min = torch.min(heights)
    z_max = torch.max(heights)
    # Calculate the threshold z-coordinate for the upper 10% of the object's height
    threshold = z_min + 0.9 * (z_max - z_min)
    # Get the indices of the points in the upper 10%
    return heights >= threshold

boundary1 = scene1.sim_obj_dict[obj_idx].set_boundary_condition("boundary1", boundary_func, bdry_penalty=10000)

def additional_event_handler(visualizer, event):
    """Event handler to be provided to Kaolin's visualizer"""
    global active_face, mesh, mouse_clicked, pull_from_2d, sim_history
    
    with visualizer.out:
        if event['shiftKey']:
            #return simplicits_viz_logic(event, visualizer, mouse_clicked, boundary1, pull_from_2d, mesh)
            if event['type'] == 'mousedown':
                mouse_clicked = 1
                pull_from_2d = torch.tensor(visualizer._get_clamped_coords(event), device='cuda', dtype=torch.float)
                current_values = visualizer.get_values_under_cursor(event)
                active_face = current_values['face_idx'][0][0]
                if active_face == -1:
                    boundary1.set_pinned_verts(None, None)
                    return False
        
                bdry_inds, bdry_pts = find_closest_3d_points(mesh.vertices[mesh.faces[active_face],:], scene1.get_object_deformed_pts(obj_idx, scene1.sim_obj_dict[obj_idx].sim_pts).squeeze(), radius=0.2, k=10)
                
                boundary1.set_pinned_verts(bdry_inds, bdry_pts)
                visualizer.render_update()
                
            if event['type'] == 'mousemove':
                if mouse_clicked == 1:
                    visualizer.out.clear_output()
                    pull_to_2d = torch.tensor(visualizer._get_clamped_coords(event), device='cuda', dtype=torch.float)
                    pull_to_2d = torch.tensor(visualizer._get_clamped_coords(event), device='cuda', dtype=torch.float)

                    # Delta for clicked point
                    pxl_offset = (pull_to_2d - pull_from_2d)

                    # pxl_offset[0] /= visualizer.width 
                    # pxl_offset[1] /= visualizer.height

                    pinned_verts = boundary1.pinned_vertices
                    point = pinned_verts.mean(dim=0) # in 3D world coord
                    # Convert to opengl sign convention
                    pxl_offset[1] *= -1
                    
                    # Get updated location of bdry - TODO: (Or Perel) Need to fix this code
                    updated_point = convert_offset_to_world_coords(visualizer.camera, point, pxl_offset)
                    offset_3d = updated_point - point
                    pinned_verts += offset_3d
                    boundary1.update_pinned(pinned_verts)
                    print(f'point:{point}')
                    print(f'updated_pt:{updated_point}')
                    print(f'offset:{offset_3d}')
                
            if event['type'] == 'mouseup':
                if mouse_clicked == 1:
                    mouse_clicked = 0
                    
            return False
        return True

interactive_cam = kal.render.easy_render.default_camera(resolution).cuda()
interactive_cam.extrinsics.move_forward(1)
interactive_viz = kal.visualize.IpyTurntableVisualizer(
    resolution, resolution, copy.deepcopy(interactive_cam), render, fast_render=fast_render,
    max_fps=10, world_up_axis=1,
    additional_event_handler=additional_event_handler,
    additional_watched_events=['mousedown', 'mousemove', 'keydown'] # We need to now watch for key press event
)
interactive_viz.render_update()

global sim_thread_open, sim_thread
sim_thread_open = False
sim_thread = None

def run_sim():
    # if shift is pressed, run our logic, else run default logic
    for step in range(1000):
        with interactive_viz.out:
            scene1.run_sim_step()
            sim_history.append(scene1.get_object_deformed_pts(obj_idx).squeeze())
            print('.', end='')
        mesh.vertices = sim_history[-1]
        interactive_viz.render_update()    

# Run sim for num_steps in a different thread
# Needed for interactivity
def callback(b):
    global sim_thread_open, sim_thread
    with interactive_viz.out:
        if(sim_thread_open):
            sim_thread.join()
            sim_thread_open = False
        sim_thread_open = True
        sim_thread = threading.Thread(target=run_sim, daemon=True)
        sim_thread.start()
        
button = Button(description='Run Test')
button.on_click(callback)
display(HBox([interactive_viz.canvas, button]), interactive_viz.out)

Output()